# RockIT · Final analysis, model training and backend exports
**Run all on Colab CPU.** Uses `RockIT/data/regional_monthly_multi.csv`, `RockIT/metadata/metadata_multi.json` and `regions_multi.geojson` from the finalized extraction. No NASA downloads here.

Stages: self-tests → data quality → trends/FDR → charts/robustness → discoveries → **required K-means training/evaluation** → exports. Overall stage progress plus individual task bars show elapsed time/ETA. Package installation uses pip's own output. Stage ETA is approximate because stages differ in cost.

Outputs are saved to a unique `RockIT/outputs/<run_id>/` folder, including `backend_bundle.zip` and `model_bundle.zip`. No synthetic data is included. Files share a run ID and input checksum.

Whole California is a validation region: by default it is excluded from cluster fitting, then assigned with `predict`, avoiding its duplicate representation alongside the Central Valley. It is not an independent accuracy test. Other geographic overlap remains. Training requires at least 20 complete non-validation fingerprints; missing/degenerate series are explicitly reported and may block training.

A fitted model is saved even if validation is weak, marked **exploratory_failed_checks**. Backend consumers must respect the model status; passing heuristic checks is not proof of natural environmental classes or forecasting ability. No causation claims. Review source/coverage reports and all charts before presenting findings.

In [1]:
%pip -q install numpy pandas scipy matplotlib statsmodels scikit-learn joblib tqdm

In [2]:
from pathlib import Path
import json, hashlib, platform, warnings, time, zipfile, shutil
from tqdm.auto import tqdm
from IPython.display import display
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
from statsmodels.stats.multitest import multipletests
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, pairwise_distances
import joblib
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
ROOT = Path('/content/drive/MyDrive/RockIT') # EDIT if needed
CONFIG = dict(start='2001-01', end='2024-12', alpha=0.05,
              min_temporal_coverage=0.90, min_spatial_coverage=0.90,
              min_per_calendar_month=15, max_gap_months=12,
              hac_lags=12, sensitivity_lags=[6,12,24], trim_years=2,
              min_cluster_regions=20, validation_regions=['california'], seed=42)
# Coverage thresholds and lag choices are proposed project defaults, not NASA standards.
# Freeze them before scanning findings; document any changes.
VARIABLES = ['surface_temperature','precipitation','soil_moisture','evapotranspiration']
EXPECTED = pd.period_range(CONFIG['start'], CONFIG['end'], freq='M')
INPUT = ROOT/'data/regional_monthly_multi.csv'
METADATA_PATH = ROOT/'metadata/metadata_multi.json'
BOUNDARY_PATH = ROOT/'metadata/regions_multi.geojson'
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ROOT/'outputs'/RUN
plt.rcParams.update({'figure.figsize':(11,4),'figure.dpi':120,'axes.grid':True,'grid.alpha':0.2})
print('Input:', INPUT, '\nAvailable:', INPUT.exists())
if not INPUT.exists():raise FileNotFoundError(f'Expected multi-region CSV: {INPUT}')
if not METADATA_PATH.exists():raise FileNotFoundError(f'Expected source metadata: {METADATA_PATH}')
if not BOUNDARY_PATH.exists():raise FileNotFoundError(f'Expected region geometry: {BOUNDARY_PATH}')
OVERALL=tqdm(total=7,desc='Overall stages',unit='stage')
def stage_done(name):
    OVERALL.set_postfix(completed=name);OVERALL.update(1)
print('Clustering training is required. California is assigned after fitting the other regions.')


Mounted at /content/drive
Input: /content/drive/MyDrive/RockIT/data/regional_monthly_multi.csv 
Available: True


Overall stages:   0%|          | 0/7 [00:00<?, ?stage/s]

Clustering training is required. California is assigned after fitting the other regions.


## 1 · Trend engine
OLS with an intercept, elapsed time in decades, and 11 calendar-month indicators. Calendar-aware Bartlett/Newey–West sandwich covariance uses actual month separations, including gaps; finite-sample correction and approximate Student-t inference are used. This accounts for residual serial correlation within the configured lag window, not all possible dependence. Confidence intervals are pointwise and do not include GLDAS structural uncertainty. FDR-adjusted decisions are separate from pointwise intervals.

Monthly climatology subtraction is for charts. Inference uses the full regression, avoiding inconsistencies from regressing only seasonally demeaned values when coverage is unbalanced.

In [3]:
def longest_gap(mask):
    best = run = 0
    for missing in ~np.asarray(mask, dtype=bool):
        run = run + 1 if missing else 0
        best = max(best, run)
    return best

def analyze_trend(dates, values, unit='', start=None, end=None, lags=None):
    periods = pd.DatetimeIndex(pd.to_datetime(dates)).to_period('M')
    if periods.has_duplicates:
        raise ValueError('Duplicate months in series')
    values = np.asarray(values, dtype=float)
    if len(values) != len(periods):
        raise ValueError('Dates and values differ in length')
    idx = pd.period_range(start or CONFIG['start'], end or CONFIG['end'], freq='M')
    s = pd.Series(values, index=periods).reindex(idx)
    valid = np.isfinite(s.to_numpy())
    counts = pd.Series(valid, index=idx).groupby(idx.month).sum()
    base = dict(unit=unit, n_observations=int(valid.sum()), expected_months=len(idx),
                coverage=float(valid.mean()), max_gap_months=longest_gap(valid),
                start=str(idx[0]), end=str(idx[-1]))
    reasons=[]
    if valid.mean() < CONFIG['min_temporal_coverage']: reasons.append('temporal coverage')
    if longest_gap(valid) > CONFIG['max_gap_months']: reasons.append('long missing interval')
    # Scale month-count requirement for intentionally shorter sensitivity windows.
    required = min(CONFIG['min_per_calendar_month'], max(3, len(idx)//12-2))
    if len(counts)!=12 or counts.min()<required: reasons.append('calendar-month coverage')
    if valid.sum()<36: reasons.append('fewer than 36 observations')
    if reasons: return dict(base,status='insufficient_data',reason='; '.join(reasons)), None
    ordinals = np.array([p.ordinal for p in idx])
    time = (ordinals-ordinals[0])/120.0
    X = np.column_stack([np.ones(len(idx)),time]+[(idx.month==m).astype(float) for m in range(2,13)])
    x,y = X[valid],s.to_numpy()[valid]
    if np.linalg.matrix_rank(x)<x.shape[1]:
        return dict(base,status='insufficient_data',reason='rank deficient design'),None
    beta = np.linalg.lstsq(x,y,rcond=None)[0]
    resid = y-x@beta
    if np.std(resid) < 1e-12*max(1,np.std(y)):
        return dict(base,status='degenerate',reason='zero or near-zero residual variance'),None
    scores = np.zeros_like(X); scores[valid] = x*resid[:,None]
    L = CONFIG['hac_lags'] if lags is None else int(lags)
    L = min(L,len(idx)-1)
    meat = scores.T@scores
    for lag in range(1,L+1):
        cross = scores[lag:].T@scores[:-lag]
        meat += (1-lag/(L+1))*(cross+cross.T)
    bread = np.linalg.inv(x.T@x)
    dof=len(y)-x.shape[1]
    cov = bread@meat@bread * len(y)/dof
    se=float(np.sqrt(max(0,cov[1,1])))
    if se<=0: return dict(base,status='degenerate',reason='zero trend uncertainty'),None
    p=float(2*student_t.sf(abs(beta[1]/se),dof))
    critical=student_t.ppf(1-CONFIG['alpha']/2,dof)
    ci=[float(beta[1]-critical*se),float(beta[1]+critical*se)]
    climatology=s.groupby(s.index.month).transform('mean')
    anomaly=s-climatology
    scale=float(anomaly.std(ddof=1))
    r=dict(base,status='ok',slope_per_decade=float(beta[1]),ci_low=ci[0],ci_high=ci[1],
           confidence_interval=ci,p_value=p,direction='increasing' if beta[1]>0 else 'decreasing',
           standardized_trend=float(beta[1]/scale) if scale>1e-12 else None,
           fitted_period_change=float(beta[1]*(time[-1]-time[0])),hac_lags=L,
           method='OLS + month indicators; calendar-aware Bartlett HAC; approximate t inference')
    details=pd.DataFrame({'date':idx.to_timestamp(),'value':s.to_numpy(),
        'anomaly':anomaly.to_numpy(),'fitted':X@beta,'residual':np.where(valid,s.to_numpy()-X@beta,np.nan),
        'trend_component':beta[1]*(time-time.mean())})
    return r,details

## 2 · Synthetic self-tests (never exported as NASA data)

In [4]:
def run_self_tests():
    dates=pd.date_range('2001-01-01','2024-12-01',freq='MS')
    rng=np.random.default_rng(42)
    tm=np.arange(len(dates))/120
    noise=rng.normal(0,0.15,len(dates))
    seasonal=3*np.sin(2*np.pi*np.arange(len(dates))/12)
    for slope in [2.0,-2.0]:
        y=10+slope*tm+seasonal+noise
        r,_=analyze_trend(dates,y)
        assert r['status']=='ok' and abs(r['slope_per_decade']-slope)<0.1
        assert r['ci_low']<slope<r['ci_high']
    y=10+2*tm+seasonal+noise; y[60:64]=np.nan
    r,_=analyze_trend(dates,y)
    assert r['status']=='ok' and abs(r['slope_per_decade']-2)<0.1
    r,_=analyze_trend(dates,np.ones(len(dates)))
    assert r['status']=='degenerate'
    y[:200]=np.nan
    assert analyze_trend(dates,y)[0]['status']=='insufficient_data'
    try: analyze_trend([dates[0],dates[0]],[1,2])
    except ValueError: pass
    else: raise AssertionError('Duplicates accepted')
    # Stationary AR(1) null: smoke check empirical rejection/interval coverage.
    rejected=0; covered=0; trials=100
    for _ in tqdm(range(trials),desc='Statistical self-tests',unit='simulation'):
        e=np.zeros(600); innovations=rng.normal(size=600)
        for j in range(1,600): e[j]=0.6*e[j-1]+innovations[j]
        r,_=analyze_trend(dates,seasonal+e[-len(dates):])
        rejected+=r['p_value']<0.05
        covered+=r['ci_low']<=0<=r['ci_high']
    assert rejected/trials<0.20, 'Unexpected null rejection rate'
    print(f'All self-tests passed. AR(1) null rejection={rejected/trials:.2f}; interval coverage={covered/trials:.2f}.')
    print('This smoke test does not validate every real-data dependence structure.')
run_self_tests()
stage_done("self-tests")


Statistical self-tests:   0%|          | 0/100 [00:00<?, ?simulation/s]

All self-tests passed. AR(1) null rejection=0.11; interval coverage=0.89.
This smoke test does not validate every real-data dependence structure.


## 3 · Load and validate real measurements
No file → subsequent real-data sections skip. Missing measurements stay missing. Invalid spatial coverage is rejected; measurements below the spatial threshold are excluded and logged. Infinite values, unknown variables, conflicting units, duplicate months, and nonnumeric values trigger an error. No automatic interpolation or unit guessing.

In [5]:
df=None; trends=pd.DataFrame(); details_by_series={}; metadata={}; warnings_list=[]
if INPUT.exists():
    df=pd.read_csv(INPUT)
    required={'date','region','variable','value','unit','valid_area_fraction'}
    if required-set(df): raise ValueError(f'Missing columns: {required-set(df)}')
    if df.empty: raise ValueError('CSV is empty')
    for col in ['region','variable','unit']:
        if df[col].isna().any(): raise ValueError(f'Missing {col}')
        df[col]=df[col].astype(str).str.strip()
        if (df[col]=='').any(): raise ValueError(f'Blank {col}')
    if set(df.variable)-set(VARIABLES): raise ValueError('Unknown variable identifiers')
    df['date']=pd.to_datetime(df.date,errors='raise')
    df['month']=df.date.dt.to_period('M')
    if df.duplicated(['region','variable','month']).any(): raise ValueError('Duplicate region-variable-month')
    for col in ['value','valid_area_fraction']: df[col]=pd.to_numeric(df[col],errors='raise')
    if np.isinf(df.value).any(): raise ValueError('Infinite measurements')
    if not df.valid_area_fraction.between(0,1).all(): raise ValueError('Coverage must be finite and between 0 and 1')
    outside=~df.month.isin(EXPECTED)
    if outside.any(): warnings_list.append(f'Excluded {outside.sum()} rows outside analysis period')
    df=df.loc[~outside].copy()
    if df.empty: raise ValueError('No rows in analysis period')
    df['analysis_value']=df.value
    df['analysis_unit']=''
    unit_aliases={'k':'K','kelvin':'K','kg/m2':'kg/m2','kg m-2':'kg/m2','kg/m^2':'kg/m2',
                  'kg/m2/s':'kg/m2/s','kg m-2 s-1':'kg/m2/s','kg/m^2/s':'kg/m2/s'}
    for variable,g in tqdm(list(df.groupby('variable')),desc='Validate and convert units',unit='variable'):
        native=g.unit.str.lower().map(unit_aliases)
        expected='K' if variable=='surface_temperature' else ('kg/m2' if variable=='soil_moisture' else 'kg/m2/s')
        if native.isna().any() or not (native==expected).all():
            raise ValueError(f'{variable}: expected native units {expected}; received {g.unit.unique()}')
        ix=g.index
        if variable=='surface_temperature':
            df.loc[ix,'analysis_value']=g.value-273.15; unit='degC'
        elif variable=='soil_moisture': unit='kg/m2'
        else:
            df.loc[ix,'analysis_value']=g.value*86400; unit='mm/day'
        df.loc[ix,'analysis_unit']=unit
    df['excluded_spatial']=df.valid_area_fraction<CONFIG['min_spatial_coverage']
    df.loc[df.excluded_spatial,'analysis_value']=np.nan
    meta_path=METADATA_PATH
    if meta_path.exists(): metadata=json.loads(meta_path.read_text())
    else: warnings_list.append('Source metadata missing: exports are provisional; do not present as verified findings.')
    OUT.mkdir(parents=True,exist_ok=True)
    print('Rows:',len(df),'Regions:',df.region.nunique())
    print('Warnings:',warnings_list)
    display(df.head())
else:
    print('Waiting for Khushi: place regional_monthly.csv in RockIT/data, then Run all.')
if df is not None:
    expected_keys=pd.MultiIndex.from_product([sorted(df.region.unique()),VARIABLES],names=['region','variable'])
    quality_input=df.groupby(['region','variable']).agg(rows=('value','size'),missing_native=('value',lambda x:x.isna().sum()),spatial_excluded=('excluded_spatial','sum'))
    quality_input=quality_input.reindex(expected_keys,fill_value=0)
    if not np.isfinite(df.loc[df.value.notna(),'value']).all():raise ValueError('Nonfinite native values')
    metadata_regions={r['region_id'] for r in metadata.get('regions',[])}
    if metadata_regions and metadata_regions!=set(df.region.unique()):raise ValueError('CSV and metadata region IDs differ')
    boundary_data=json.loads(BOUNDARY_PATH.read_text())
    boundary_ids={f['properties']['region_id'] for f in boundary_data['features']}
    if boundary_ids!=set(df.region.unique()):raise ValueError('CSV and geometry region IDs differ')
    display(quality_input)
stage_done('data validation')


Validate and convert units:   0%|          | 0/4 [00:00<?, ?variable/s]

Rows: 24192 Regions: 21
Warnings: []


,date,region,variable,value,unit,valid_area_fraction,month,analysis_value,analysis_unit,excluded_spatial
0,2001-01-01,alaska_tundra,evapotranspiration,0.000001,kg m-2 s-1,1.0,2001-01,0.092958,mm/day,False
1,2001-01-01,alaska_tundra,precipitation,0.000007,kg m-2 s-1,1.0,2001-01,0.588687,mm/day,False
2,2001-01-01,alaska_tundra,soil_moisture,23.753559,kg m-2,1.0,2001-01,23.753559,kg/m2,False
3,2001-01-01,alaska_tundra,surface_temperature,250.359688,K,1.0,2001-01,-22.790312,degC,False
4,2001-01-01,alps,evapotranspiration,0.000001,kg m-2 s-1,1.0,2001-01,0.113864,mm/day,False


rows  missing_native  spatial_excluded
region         variable                                                   
alaska_tundra  surface_temperature   288               0                 0
               precipitation         288               0                 0
               soil_moisture         288               0                 0
               evapotranspiration    288               0                 0
alps           surface_temperature   288               0                 0
...                                  ...             ...               ...
siberian_taiga evapotranspiration    288               0                 0
yangtze        surface_temperature   288               0                 0
               precipitation         288               0                 0
               soil_moisture         288               0                 0
               evapotranspiration    288               0                 0

[84 rows x 3 columns]

## 4 · All trends, quality report and FDR correction
The test family is every eligible region–variable trend in this run. Adjusted p-values use Benjamini–Hochberg. Spatial dependence can affect FDR guarantees; results remain exploratory. An inconclusive trend is not proof of no change.

In [6]:
if df is not None:
    rows=[]
    for region in tqdm(sorted(df.region.unique()),desc='Trends: four variables per region',unit='region'):
        for variable in VARIABLES:
            g=df[(df.region==region)&(df.variable==variable)]
            unit=g.analysis_unit.iloc[0] if len(g) else ''
            r,d=analyze_trend(g.date,g.analysis_value,unit=unit)
            r.update(region=region,variable=variable,spatial_excluded_rows=int(g.excluded_spatial.sum()))
            rows.append(r)
            if d is not None: details_by_series[(region,variable)]=d
    trends=pd.DataFrame(rows)
    trends['q_value']=np.nan; trends['significant']=False
    ok=trends.status.eq('ok')
    if ok.any():
        reject,q,_,_=multipletests(trends.loc[ok,'p_value'],alpha=CONFIG['alpha'],method='fdr_bh')
        trends.loc[ok,'q_value']=q; trends.loc[ok,'significant']=reject
    trends['evidence']='insufficient_data'
    trends.loc[ok,'evidence']='inconclusive'
    trends.loc[ok & trends.significant,'evidence']='supported_'+trends.loc[ok & trends.significant,'direction']
    display(trends)

stage_done("trends and FDR")


Trends: four variables per region:   0%|          | 0/21 [00:00<?, ?region/s]

,unit,n_observations,expected_months,coverage,max_gap_months,start,end,status,slope_per_decade,ci_low,...,standardized_trend,fitted_period_change,hac_lags,method,region,variable,spatial_excluded_rows,q_value,significant,evidence
0,degC,288,288,1.0,0,2001-01,2024-12,ok,-0.231658,-0.935331,...,-0.082484,-0.554050,12,OLS + month indicators; calendar-aware Bartlet...,alaska_tundra,surface_temperature,0,6.362987e-01,False,inconclusive
1,mm/day,288,288,1.0,0,2001-01,2024-12,ok,0.104566,0.044602,...,0.284104,0.250087,12,OLS + month indicators; calendar-aware Bartlet...,alaska_tundra,precipitation,0,3.215922e-03,True,supported_increasing
2,kg/m2,288,288,1.0,0,2001-01,2024-12,ok,0.379374,-0.626917,...,0.108707,0.907336,12,OLS + month indicators; calendar-aware Bartlet...,alaska_tundra,soil_moisture,0,5.836895e-01,False,inconclusive
3,mm/day,288,288,1.0,0,2001-01,2024-12,ok,0.045934,0.013392,...,0.344732,0.109860,12,OLS + month indicators; calendar-aware Bartlet...,alaska_tundra,evapotranspiration,0,1.884527e-02,True,supported_increasing
4,degC,288,288,1.0,0,2001-01,2024-12,ok,0.774427,0.399435,...,0.462133,1.852170,12,OLS + month indicators; calendar-aware Bartlet...,alps,surface_temperature,0,3.756159e-04,True,supported_increasing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,mm/day,288,288,1.0,0,2001-01,2024-12,ok,0.065739,0.038213,...,0.543919,0.157225,12,OLS + month indicators; calendar-aware Bartlet...,siberian_taiga,evapotranspiration,0,2.642338e-05,True,supported_increasing
80,degC,288,288,1.0,0,2001-01,2024-12,ok,0.597000,0.455564,...,0.609420,1.427826,12,OLS + month indicators; calendar-aware Bartlet...,yangtze,surface_temperature,0,7.463678e-14,True,supported_increasing
81,mm/day,288,288,1.0,0,2001-01,2024-12,ok,0.204218,0.026316,...,0.283290,0.488422,12,OLS + month indicators; calendar-aware Bartlet...,yangtze,precipitation,0,6.669701e-02,False,inconclusive
82,kg/m2,288,288,1.0,0,2001-01,2024-12,ok,0.277275,-0.109765,...,0.232589,0.663150,12,OLS + month indicators; calendar-aware Bartlet...,yangtze,soil_moisture,0,2.680850e-01,False,inconclusive


## 5 · Charts and robustness
Annual values below are inspection summaries requiring all 12 valid months, not inputs to monthly inference. Lag and endpoint sensitivity are diagnostics, not extra opportunities to select a preferred p-value. Robustness does not mean independent replication. Large standardized residuals are flagged but retained.

In [7]:
sensitivity=[]
if df is not None:
    (OUT/'plots').mkdir(exist_ok=True)
    for (region,variable),d in tqdm(list(details_by_series.items()),desc='Charts + five robustness checks per series',unit='series'):
        r=trends[(trends.region==region)&(trends.variable==variable)].iloc[0]
        safe=hashlib.sha256(f'{region}|{variable}'.encode()).hexdigest()[:12]
        fig,ax=plt.subplots(3,1,figsize=(11,10),layout='constrained')
        ax[0].plot(d.date,d.value,label='Monthly measurement'); ax[0].plot(d.date,d.fitted,label='Seasonal fit + trend')
        ax[0].set_ylabel(r.unit); ax[0].legend()
        ax[1].plot(d.date,d.anomaly,label='Monthly anomaly'); ax[1].plot(d.date,d.trend_component,label='Trend component')
        ax[1].set_ylabel(r.unit); ax[1].legend()
        annual=d.set_index('date').value.resample('YS').agg(['mean','count'])
        ax[2].plot(annual.index,annual['mean'].where(annual['count']==12),marker='o')
        ax[2].set_ylabel(r.unit); ax[2].set_title('Annual means: complete years only')
        fig.suptitle(f'{region} · {variable} | {r.slope_per_decade:.3g} {r.unit}/decade | q={r.q_value:.3g}')
        fig.savefig(OUT/'plots'/f'{safe}.png'); plt.close(fig)
        for label,start,end,lag in [('lag_'+str(L),CONFIG['start'],CONFIG['end'],L) for L in CONFIG['sensitivity_lags']]+[
            ('trim_start',str(EXPECTED[0]+12*CONFIG['trim_years']),CONFIG['end'],CONFIG['hac_lags']),
            ('trim_end',CONFIG['start'],str(EXPECTED[-1]-12*CONFIG['trim_years']),CONFIG['hac_lags'])]:
            sr,_=analyze_trend(d.date,d.value,unit=r.unit,start=start,end=end,lags=lag)
            sensitivity.append(dict(region=region,variable=variable,check=label,**sr))
    robustness=pd.DataFrame(sensitivity)
    display(robustness[[c for c in ['region','variable','check','status','slope_per_decade','p_value'] if c in robustness]])
    print('Charts saved:',OUT/'plots')
stage_done("charts and robustness")


Charts + five robustness checks per series:   0%|          | 0/84 [00:00<?, ?series/s]

,region,variable,check,status,slope_per_decade,p_value
0,alaska_tundra,surface_temperature,lag_6,ok,-0.231658,0.504376
1,alaska_tundra,surface_temperature,lag_12,ok,-0.231658,0.517462
2,alaska_tundra,surface_temperature,lag_24,ok,-0.231658,0.536926
3,alaska_tundra,surface_temperature,trim_start,ok,-0.437537,0.190024
4,alaska_tundra,surface_temperature,trim_end,ok,-0.244876,0.611524
...,...,...,...,...,...,...
415,yangtze,evapotranspiration,lag_6,ok,0.034747,0.107426
416,yangtze,evapotranspiration,lag_12,ok,0.034747,0.157808
417,yangtze,evapotranspiration,lag_24,ok,0.034747,0.191434
418,yangtze,evapotranspiration,trim_start,ok,0.016192,0.557314


Charts saved: /content/drive/MyDrive/RockIT/outputs/20261114T211652769653Z/plots


## 6 · Hotspots, Earth Opposites and Earth Twins
Hotspots are ranked within variables, retaining physical units and uncertainty. Opposites require both FDR-supported directions; this does not test the difference between their slopes. Twins use Euclidean distance between trend/anomaly-standard-deviation fingerprints: similar historical changes, not identical climate. Fingerprints include uncertain trends, so matches are exploratory. No forced discovery if criteria are unmet.

In [8]:
discoveries={'hotspots':[],'opposites':[],'twins':[]}; fingerprints=pd.DataFrame()
if not trends.empty and 'standardized_trend' in trends:
    eligible=trends[trends.status.eq('ok')].copy()
    supported=eligible[eligible.significant]
    for variable,g in supported.groupby('variable'):
        discoveries['hotspots']+=g.assign(magnitude=g.slope_per_decade.abs()).sort_values('magnitude',ascending=False).head(5).to_dict('records')
        down=g[g.slope_per_decade<0].sort_values('slope_per_decade')
        up=g[g.slope_per_decade>0].sort_values('slope_per_decade',ascending=False)
        if len(down) and len(up):
            discoveries['opposites'].append({'variable':variable,'decreasing':down.iloc[0].to_dict(),'increasing':up.iloc[0].to_dict()})
    fingerprints=eligible.pivot(index='region',columns='variable',values='standardized_trend').reindex(columns=VARIABLES).replace([np.inf,-np.inf],np.nan).dropna()
    if len(fingerprints)>=2:
        distance=pairwise_distances(fingerprints); np.fill_diagonal(distance,np.inf)
        pairs=sorted((float(distance[i,j]),i,j) for i in range(len(distance)) for j in range(i+1,len(distance)))
        discoveries['twins']=[{'region_a':str(fingerprints.index[i]),'region_b':str(fingerprints.index[j]),'distance':v} for v,i,j in pairs[:10]]
    display(pd.DataFrame(discoveries['twins']))
stage_done("discovery matching")


,region_a,region_b,distance
0,alaska_tundra,kazakh_steppe,0.085726
1,arabian_desert,sahara,0.135829
2,california,central_valley,0.157685
3,great_plains,mississippi,0.187767
4,alps,gangetic_plains,0.270560
5,great_victoria_desert,pampas,0.297376
6,canadian_boreal,great_plains,0.335161
7,great_plains,pampas,0.344724
8,canadian_boreal,pampas,0.356724
9,amazon,congo,0.357844


## 7 · Required clustering: fit, evaluate, save
Each training sample is a regional four-variable standardized trend fingerprint. The scaling is physical slope per decade divided by local monthly-anomaly standard deviation; no new StandardScaler is silently applied.

Fit K-means for k=2–5 with 30 initializations. For each, assess silhouette, minimum group size, reproducibility over three seeds and agreement with two endpoint-trimmed feature sets. Heuristic checks: silhouette ≥0.25, minimum group size ≥3, seed ARI ≥0.8 and endpoint ARI ≥0.7. Endpoint scores compare clustering of re-estimated fingerprints; they are sensitivity diagnostics, not prediction accuracy.

Prefer the highest-silhouette model among passing candidates; if none passes, retain the highest-silhouette fit as explicitly exploratory. No fabricated labels or claimed classification accuracy. Identical/insufficient fingerprints stop training with a clear error. California is assigned after fitting and is excluded from evaluation metrics.

In [9]:
cluster_results=pd.DataFrame();cluster_scores=pd.DataFrame();model_bundle=None
training_fp=fingerprints.drop(index=CONFIG['validation_regions'],errors='ignore').copy()
if len(training_fp)<CONFIG['min_cluster_regions']:
    report=trends[['region','variable','status']].copy()
    report.to_csv(OUT/'training_blockers.csv',index=False)
    raise ValueError(f'Required clustering needs {CONFIG["min_cluster_regions"]} complete non-validation fingerprints; got {len(training_fp)}. See training_blockers.csv. Do not duplicate samples or fill missing trends with zero.')
X=training_fp.to_numpy(dtype=float)
unique_count=len(np.unique(X,axis=0))
if unique_count<2:raise ValueError('Regional fingerprints are identical; clustering is undefined')
ks=list(range(2,min(5,len(X)-1,unique_count)+1))
candidates=[];models={}
with tqdm(total=6*len(ks),desc='Train/evaluate K-means',unit='fit') as progress:
    for k in ks:
        model=KMeans(n_clusters=k,n_init=30,random_state=CONFIG['seed']).fit(X)
        labels=model.labels_;progress.update(1);progress.set_postfix(k=k)
        observed=len(set(labels))
        if observed<2:
            progress.update(5);continue
        silhouette=float(silhouette_score(X,labels))
        seed_scores=[]
        for seed in [7,19,73]:
            other=KMeans(n_clusters=k,n_init=30,random_state=seed).fit_predict(X)
            seed_scores.append(float(adjusted_rand_score(labels,other)));progress.update(1)
        endpoint_scores={}
        for check in ['trim_start','trim_end']:
            sub=robustness[(robustness.check==check)&robustness.status.eq('ok')]
            fp=sub.pivot(index='region',columns='variable',values='standardized_trend').reindex(index=training_fp.index,columns=VARIABLES)
            if np.isfinite(fp.to_numpy()).all() and len(np.unique(fp.to_numpy(),axis=0))>=k:
                alt=KMeans(n_clusters=k,n_init=30,random_state=CONFIG['seed']).fit_predict(fp.to_numpy())
                endpoint_scores[check]=float(adjusted_rand_score(labels,alt))
            else:endpoint_scores[check]=None
            progress.update(1)
        endpoint_values=[v for v in endpoint_scores.values() if v is not None]
        endpoint_ari=min(endpoint_values) if len(endpoint_values)==2 else np.nan
        minimum=int(np.bincount(labels,minlength=k).min())
        passed=bool(observed==k and minimum>=3 and silhouette>=0.25 and min(seed_scores)>=0.8 and np.isfinite(endpoint_ari) and endpoint_ari>=0.7)
        candidates.append(dict(k=k,silhouette=silhouette,min_group_size=minimum,seed_ari=min(seed_scores),
            endpoint_ari=endpoint_ari,trim_start_ari=endpoint_scores['trim_start'],trim_end_ari=endpoint_scores['trim_end'],passes_checks=passed))
        models[k]=model
if not candidates:raise ValueError('No usable clustering fit; inspect fingerprints')
cluster_scores=pd.DataFrame(candidates)
passing=cluster_scores[cluster_scores.passes_checks]
selection=passing if len(passing) else cluster_scores
best=selection.sort_values('silhouette',ascending=False).iloc[0]
model=models[int(best.k)]
MODEL_STATUS='passed_heuristic_checks' if bool(best.passes_checks) else 'exploratory_failed_checks'
assigned=model.predict(fingerprints.to_numpy(dtype=float))
cluster_results=pd.DataFrame({'region':fingerprints.index,'cluster':assigned,
    'role':['validation_assignment' if r in CONFIG['validation_regions'] else 'training' for r in fingerprints.index]})
cluster_results['distance_to_center']=np.linalg.norm(fingerprints.to_numpy()-model.cluster_centers_[assigned],axis=1)
cluster_results['model_status']=MODEL_STATUS
model_bundle={'model':model,'features':VARIABLES,'scaling':'physical trend per decade / local monthly-anomaly standard deviation',
    'config':CONFIG,'run_id':RUN,'model_status':MODEL_STATUS,'training_regions':training_fp.index.tolist(),
    'evaluation':best.to_dict(),'input_sha256':hashlib.sha256(INPUT.read_bytes()).hexdigest()}
cluster_payload={'schema_version':'1.0','run_id':RUN,'status':MODEL_STATUS,'algorithm':'KMeans',
    'n_training_regions':len(training_fp),'k':int(best.k),'features':VARIABLES,
    'scaling':model_bundle['scaling'],'evaluation':best.to_dict(),'candidates':cluster_scores.to_dict('records'),
    'centers':[{'cluster':i,'features':dict(zip(VARIABLES,map(float,center)))} for i,center in enumerate(model.cluster_centers_)],
    'assignments':cluster_results.to_dict('records'),
    'limitations':['Descriptive grouping, not forecasts or causal effects','Overlapping regions are not independent samples','Cluster IDs are arbitrary; stability cutoffs are project heuristics']}
display(cluster_scores);display(cluster_results)
print('TRAINING COMPLETE:',MODEL_STATUS,'| k =',int(best.k),'| training samples =',len(training_fp))
stage_done('model fitted and evaluated')


Train/evaluate K-means:   0%|          | 0/24 [00:00<?, ?fit/s]

,k,silhouette,min_group_size,seed_ari,endpoint_ari,trim_start_ari,trim_end_ari,passes_checks
0,2,0.327810,7,1.000000,0.800089,0.800089,1.000000,True
1,3,0.254884,6,0.602978,0.627792,1.000000,0.627792,False
2,4,0.253021,2,0.386827,0.135984,0.610005,0.135984,False
3,5,0.318253,2,0.859908,0.534402,0.684793,0.534402,False


,region,cluster,role,distance_to_center,model_status
0,alaska_tundra,0,training,0.590547,passed_heuristic_checks
1,alps,0,training,0.466076,passed_heuristic_checks
2,amazon,0,training,0.569358,passed_heuristic_checks
3,arabian_desert,1,training,0.153617,passed_heuristic_checks
4,california,0,validation_assignment,0.369547,passed_heuristic_checks
5,canadian_boreal,0,training,0.537696,passed_heuristic_checks
6,central_andean_puna,1,training,0.790645,passed_heuristic_checks
7,central_valley,0,training,0.337548,passed_heuristic_checks
8,congo,0,training,0.571903,passed_heuristic_checks
9,ethiopian_highlands,1,training,0.426979,passed_heuristic_checks


TRAINING COMPLETE: passed_heuristic_checks | k = 2 | training samples = 20


## 8 · Backend bundle and model bundle
The backend consumes precomputed JSON; it does not need to train or load joblib for normal page requests.

`backend_bundle.zip` contains `trends.json`, `chart_data.json`, `discoveries.json`, `clusters.json`, `regions.geojson`, `manifest.json`, `README.md`. Every analytical payload has the same run ID and schema version. The frontend must display units, uncertainty, adjusted evidence and model status. Charts preserve missing values as null. Twin distance is not a probability.

`model_bundle.zip` contains the fitted `models.joblib`, regional fingerprints, cluster assignments, candidate evaluation and manifest. Only load model files you trust. No credentials/raw NASA downloads are included.

Exports are provisional pending human review. Pointwise trend intervals are approximate HAC intervals, not simultaneous confidence intervals. FDR decisions and raw p-values are both retained. Source structural uncertainty is not captured. Do not select different endpoint settings just to obtain significance.

In [10]:
def clean_json(value):
    if isinstance(value,dict):return {str(k):clean_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [clean_json(v) for v in value]
    if isinstance(value,np.ndarray):return clean_json(value.tolist())
    if isinstance(value,np.generic):value=value.item()
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

def save_json(path,payload):
    path.write_text(json.dumps(clean_json(payload),indent=2,allow_nan=False))

BACKEND=OUT/'backend_bundle';MODEL=OUT/'model_bundle'
BACKEND.mkdir(exist_ok=True);MODEL.mkdir(exist_ok=True)
export_tasks=[('trend table',lambda:trends.to_csv(OUT/'trends.csv',index=False)),
 ('robustness table',lambda:robustness.to_csv(OUT/'robustness.csv',index=False)),
 ('measurements',lambda:df.to_csv(OUT/'validated_measurements.csv',index=False)),
 ('fingerprints',lambda:fingerprints.to_csv(MODEL/'fingerprints.csv')),
 ('cluster assignments',lambda:cluster_results.to_csv(MODEL/'region_clusters.csv',index=False)),
 ('candidate evaluations',lambda:cluster_scores.to_csv(MODEL/'cluster_evaluation.csv',index=False)),
 ('fitted model',lambda:joblib.dump(model_bundle,MODEL/'models.joblib'))]
for name,task in tqdm(export_tasks,desc='Save tables/model',unit='file'):
    task()
chart_data=[];plots=[]
for (region,variable),d in tqdm(list(details_by_series.items()),desc='Prepare chart JSON',unit='series'):
    r=trends[(trends.region==region)&(trends.variable==variable)].iloc[0]
    chart_data.append(dict(region=region,variable=variable,unit=r.unit,series=d.assign(date=d.date.dt.strftime('%Y-%m-%d')).to_dict('records')))
    safe=hashlib.sha256(f'{region}|{variable}'.encode()).hexdigest()[:12]
    plots.append(dict(region=region,variable=variable,path=f'plots/{safe}.png'))
env={p:version(p) for p in ['numpy','pandas','scipy','matplotlib','statsmodels','scikit-learn','joblib','tqdm']}
manifest=dict(schema_version='1.0',run_id=RUN,status='provisional_pending_scientific_review',model_status=MODEL_STATUS,
    config=CONFIG,source_metadata=metadata,warnings=warnings_list,
    input_sha256=hashlib.sha256(INPUT.read_bytes()).hexdigest(),python=platform.python_version(),packages=env,
    plots=plots,region_count=int(df.region.nunique()),trend_test_count=int(trends.status.eq('ok').sum()),
    significance_family='all eligible region-variable trend tests in this run',
    limitations=['Historical association, not causation or forecasting','Pointwise HAC intervals are approximate and exclude GLDAS structural uncertainty',
      'Regions overlap and differ in area/type','Fingerprint uncertainty affects similarity and clustering'])
readme=f"""RockIT backend bundle | schema 1.0 | run {RUN}
Load these JSON files at backend startup. Serve trends, series, discoveries, clusters and region geometry through your API.
All analytical payloads have run_id/schema_version; never mix different runs.
trends.json: results array; chart_data.json: series array; discoveries.json: hotspots/opposites/twins; clusters.json: centers/assignments/evaluation/status.
regions.geojson contains source geometry and name/region_id properties. manifest.json contains methods, units, sources and configuration.
Model status: {MODEL_STATUS}. Human scientific review remains required. Respect status in the frontend, including exploratory_failed_checks.
Cluster IDs are arbitrary. No supervised accuracy or future prediction is claimed. No significant trend means insufficient evidence, not proof of no change.
The joblib artifact is separately packaged in model_bundle.zip; it is not needed for displaying precomputed results.
"""
json_tasks={
 'trends.json':{'schema_version':'1.0','run_id':RUN,'results':trends.to_dict('records')},
 'chart_data.json':{'schema_version':'1.0','run_id':RUN,'series':chart_data},
 'discoveries.json':{'schema_version':'1.0','run_id':RUN,**discoveries},
 'clusters.json':cluster_payload,'manifest.json':manifest}
for name,payload in tqdm(list(json_tasks.items()),desc='Write backend JSON',unit='file'):
    save_json(BACKEND/name,payload)
# Re-serialize rather than modifying extractor's geometry file.
(BACKEND/'regions.geojson').write_text(json.dumps(boundary_data))
(BACKEND/'README.md').write_text(readme)
save_json(MODEL/'manifest.json',manifest)
save_json(OUT/'manifest.json',manifest)
quality_input.to_csv(OUT/'input_quality.csv')
# Validate strict JSON and consistent run IDs before packaging.
for name in json_tasks:
    payload=json.loads((BACKEND/name).read_text(),parse_constant=lambda x:(_ for _ in ()).throw(ValueError(x)))
    if payload['run_id']!=RUN:raise ValueError('Export run ID mismatch')
for folder in tqdm([BACKEND,MODEL],desc='Package deliverables',unit='bundle'):
    with zipfile.ZipFile(OUT/(folder.name+'.zip'),'w',zipfile.ZIP_DEFLATED) as z:
        for path in sorted(folder.iterdir()):z.write(path,path.name)
stage_done('exports verified');OVERALL.close()
print('FINISHED:',OUT)
print('BACKEND:',OUT/'backend_bundle.zip')
print('MODEL:',OUT/'model_bundle.zip')
print('Charts:',OUT/'plots')
print('Review trend uncertainty, robustness, source coverage and clustering status before publishing.')


Save tables/model:   0%|          | 0/7 [00:00<?, ?file/s]

Prepare chart JSON:   0%|          | 0/84 [00:00<?, ?series/s]

Write backend JSON:   0%|          | 0/5 [00:00<?, ?file/s]

Package deliverables:   0%|          | 0/2 [00:00<?, ?bundle/s]

FINISHED: /content/drive/MyDrive/RockIT/outputs/20261114T211652769653Z
BACKEND: /content/drive/MyDrive/RockIT/outputs/20261114T211652769653Z/backend_bundle.zip
MODEL: /content/drive/MyDrive/RockIT/outputs/20261114T211652769653Z/model_bundle.zip
Charts: /content/drive/MyDrive/RockIT/outputs/20261114T211652769653Z/plots
Review trend uncertainty, robustness, source coverage and clustering status before publishing.
